In [239]:
# Setup debugging and connection parameters
CONNECT_TO_CHATGPT_API = False
REASONING_THRESHOLD = 0.70
DEBUG_LEVEL = 0


In [240]:
# 1.  Read a list of math arithmetic problems from a JSON file
#
# For example:
# [
# {
#   "num_candidate_answers":  10,
#   "retry_max_times": 10,
#   "problems":  [
#   { "problem":  "4 * 25",
#     "answer":  "100",
#     "related_problems":  [ { "problem":  "How many quarters are in a dollar",
#                              "answer":  4
#                            }
#                          ]
#   },
#   { "problem":  "4 * 250",
#     "answer":  "1000",
#     "related_problems":  [ { "problem":  "4 * 25",
#                              "answer":  100
#                            }
#                          ]
#   },
#   { "problem":  "1000 / 4",
#     "answer":  "250",
#     "related_problems":  [ { "problem":  "4 * 250",
#                              "answer":  1000
#                            }
#                          ]
#   },
#   { "problem":  "996 / 4",
#     "answer":  "249",
#     "related_problems":  [ { "problem":  "1000 / 4",
#                              "answer":  250
#                            }
#                          ],
#     "related_properties": [ { "property":  "distributive",
#                               "usage":  "996 / 4 = (1000 - 4 ) / 4 = (1000 + (-4)) * (1/4) = (1000 * (1/4)) + ((-4) * (1/4)) = (1000 / 4) + (-4/4) = 250 - 1 = 249"
#                             }
#                           ],
#     "common_errors": [ { "error":  "996 / 4 = 6",
#                          "reasoning":  "9 + 9 + 6 = 24, 24 / 4 = 6" }
#                      ]
#   }
#   ]
# }
# ]

import pandas as pd

math_problems_df = pd.read_json('/home/myang/education/capstone-project/MYangAICapstoneProjectUMGCPrototypeInput.json')

if (DEBUG_LEVEL > 0):
  print(math_problems_df.head())


In [241]:
# 2.  Connect to the ChatGPT API

import os
from openai import OpenAI

if (CONNECT_TO_CHATGPT_API):
  client = OpenAI(
      api_key=os.environ.get("OPENAI_API_KEY"),
  )

  # Briefly test the connection
  response = client.responses.create(
      model="gpt-5.5",
      instructions="You are a math teacher assistant.",
      input="What is 2 + 2?"
  )

  if (DEBUG_LEVEL > 0):
    print(response.output_text)


In [242]:
# 3.  Iteratively looping over the math problems read from step 1, do the following steps:
#   (a)  Ask a student the math problem from the current list iteration.  For example:  996 / 4.
#   (b)  Pass the student's answer to ChatGPT and ask ChatGPT to verify if the student is correct (correct answers can
#        come in different forms, and this allows ChatGPT to resolve whether the student has provided an equivalent
#        correct answer in a different form.  For instance, 996 / 4 = 249 but it also equals "249.0 and 0/4".)
#   (c)  Whether the student was correct or incorrect in step 3(b), ask the student for what reasoning went into the answer.
#        If the answer came in a format unexpected for the question (for instance if a decimal number answer was provided when
#        a fraction was expected), ask the student why they chose a particular format for the answer.
#      (i)  If the student's answer was correct and also contains elements expected in reasoning (determined by cosine
#           similarity between the student's answer and reasoning provided for the problem in the JSON problem inputs), break
#           and go into the next loop iteration (for the next math problem in the list).
#      (ii)  If the student's answer was incorrect and/or did not contain reasoning elements expected for
#            an answer to the problem, prepare text melding the student's answer with input text from the JSON file
#            and ask ChatCPT to formulate a set of productive questions to help lead the student in a good direction.
#            Specify ask ChatGPT to provide n candidate answers to pick from, where n was specified in the JSON setup
#            file read in step 1 above.  For example:
#            “A child incorrectly thinks that 996 / 4 = 6
#             when correctly 996 / 4 = (1000 - 4 ) / 4 = (1000 + (-4)) * (1/4)
#             = (1000 * (1/4)) + ((-4) * (1/4)) = (1000 / 4) + (-4/4) = 250 - 1 = 249,
#             using the distributive property.  Ask a question that asks the child a
#             productive question (not an answer) containing correct reasoning.  Provide 10 possible
#             productive questions.  The child might have incorrectly reasoned that
#             996 / 4 = 6 because 9 + 9 + 6 = 24 and 24 / 4 = 6.”
#      (iii)  Read the ChatGPT answers and pick the "best" one to convey back to the student, using TF-IDF and cosine similarity
#             between notes in the JSON input file, text originating from the student, and the answers provided by ChatGPT.
#      (iv)  Continue steps (c)(i) through (c)(iii) until the student gives a correct answer containing at least some elements
#            of reasoning specified as desired in the JSON input file, or if the student wants to give up on the problem
#            (the correct answer should not be provided) or if a certain number of attempts t, read in from the JSON file
#             in step 1, was exceeded).

In [243]:
import re

def chatgpt_api_supplies_leading_questions(problem_df, student_answer, student_reasoning, num_leading_questions_to_supply):

  leading_question_prompt = "A child incorrectly thinks that %s = %s when correctly %s = %s" \
                            % (problem_df['problem'][0], student_answer, problem_df['problem'][0], problem_df['answer'][0])

  if ('related_properties' in problem_df.columns):
    related_properties_df = pd.DataFrame.from_dict(problem_df['related_properties']).dropna()
    if (len(related_properties_df) > 0):
      related_property = related_properties_df['related_properties'][0]
      related_property_property = related_property['property']
      related_property_usage = related_property['usage']
      leading_question_prompt += " because %s from the %s property" % (related_property_usage, related_property_property)

  leading_question_prompt += ".  Ask the child %s candidate productive questions (not answers) containing correct reasoning.  " \
                             % num_leading_questions_to_supply

  leading_question_prompt += "  The child reasoned that '%s'" % student_reasoning

  if ('common_errors' in problem_df.columns):
    common_errors_df = pd.DataFrame.from_dict(problem_df['common_errors']).dropna()
    if (len(common_errors_df) > 0):
      leading_question_prompt += ", and might have also incorrectly reasoned that: "
      for index in range(0, len(common_errors_df), 1):
        common_error = common_errors_df['common_errors'][index]
        common_error_error = common_error['error']
        common_error_reasoning = common_error['reasoning']
        if (index > 0):
         leading_question_prompt += ", and "
        leading_question_prompt += "  %s" % common_error_reasoning 
      leading_question_prompt += ".  "

  if ('related_problems' in problem_df.columns):
    related_problems_df = pd.DataFrame.from_dict(problem_df['related_problems']).dropna()
    if (len(related_problems_df) > 0):
      leading_question_prompt += "  Related problems:  "
      for index in range(0, len(related_problems_df), 1):
        leading_question_addendum = "%s = %s" %  (related_problems_df['related_problems'][index]['problem'], related_problems_df['related_problems'][index]['answer'])
        leading_question_prompt += leading_question_addendum
        leading_question_prompt += ".  "

  if ('related_properties' in problem_df.columns):
    related_properties_df = pd.DataFrame.from_dict(problem_df['related_properties']).dropna()
    if (len(related_properties_df) > 1):
      leading_question_prompt += "  Other related properties:  "
      for index in range(1, len(related_properties_df), 1):
        leading_question_addendum = "%s due to the %s property" %  (related_properties_df['related_properties'][index]['property'], related_properties_df['related_properties'][index]['usage'])
        leading_question_prompt += leading_question_addendum
        leading_question_prompt += ".  "

  if (DEBUG_LEVEL > 0):
    print("Question giving to ChatGPT:  %s" % leading_question_prompt)

  chatgpt_feedback_list = []
  response_output = "1. If (996 div 4) meant 4 equal groups, would each group having 6 make a total of (4 times 6)? Is that total close to 996?" \
                    "2. Since (1000 div 4 = 250), and 996 is 4 less than 1000, how much less should each group get when we split 996 into 4 equal groups?" \
                    "3. Can we write (996) as (1000 - 4)? Then what would ((1000 - 4) div 4) become using the distributive property?" \
                    "4. What is (1000 div 4), and what is (4 div 4)? So what should ((1000 div 4) - (4 div 4)) be?" \
                    "5. When you added (9 + 9 + 6), did you keep the place values of 996, where the first 9 means 900 and the second 9 means 90?" \
                    "6. Is (996) the same number as (9 + 9 + 6), or is (996 = 900 + 90 + 6)? How does that change the division?" \
                    "7. If (24 div 4 = 6), what number are we dividing there: 24 or 996? Are those the same amount?" \
                    "8. Could adding the digits tell us something about divisibility, but not the actual answer to (996 div 4)? How could we check the actual quotient?" \
                    "9. If each group had 249, what would (249 times 4) give? How does that help check (996 div 4)?" \
                    "10. Can we split (996) into parts that are easy to divide by 4, like (800 + 196) or (1000 - 4), and divide each part correctly?"

  if (CONNECT_TO_CHATGPT_API):  
    response = client.responses.create(model="gpt-5.5",
                                       instructions="You are a math teacher assistant.",
                                       input=leading_question_prompt
                                      )
    response_output = response.output_text

  chatgpt_feedback_list = re.split(r'[0-9]+\.[ \t]+', response_output)
  if (len(chatgpt_feedback_list) > 0):
    chatgpt_feedback_list = chatgpt_feedback_list[1:]

  best_chatgpt_feedback = pick_best_chatgpt_feedback(problem_df, student_reasoning, chatgpt_feedback_list, num_leading_questions_to_supply)

  if (DEBUG_LEVEL > 0):
    print("chatgpt_feedback_list:  %s" % chatgpt_feedback_list)
    print("best_chatgpt_feedback:  %s" % best_chatgpt_feedback)

  return best_chatgpt_feedback


In [244]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def similarity_between_student_reasoning_and_properties(problem_df, student_reasoning, previous_student_reasoning):
  # This function needs to use TF-IDF and cosine similarity to find similarities
  # between student reasoning (and a list of previous student reasoning) and
  # teacher notes in the problem_df (that were read from the problem set .JSON file).

  texts_to_correlate = [ student_reasoning ] 

  if ('related_properties' in problem_df.columns):
    related_properties_df = pd.DataFrame.from_dict(problem_df['related_properties']).dropna()
    if (len(related_properties_df) > 0):
      for index in range(1, len(related_properties_df)):
        property_text = "%s:  %s" %  (related_properties_df['related_properties'][index]['property'], related_properties_df['related_properties'][index]['usage'])
        texts_to_correlate.append(property_text)

  if ('related_problems' in problem_df.columns):
    related_problems_df = pd.DataFrame.from_dict(problem_df['related_problems']).dropna()
    if (len(related_problems_df) > 0):
      for index in range(0, len(related_problems_df), 1):
        property_text = "%s = %s" %  (related_problems_df['related_problems'][index]['problem'], related_problems_df['related_problems'][index]['answer'])
        texts_to_correlate.append(property_text)

  if (DEBUG_LEVEL > 0):
    print("Texts to correlate:  %s" % texts_to_correlate)

  vectorizer = TfidfVectorizer(min_df=2, max_df=0.7)
  vectorizer = TfidfVectorizer()
  vectorized_data = vectorizer.fit_transform(texts_to_correlate)

  tfidf_df = pd.DataFrame(vectorized_data.toarray(),
                          columns=vectorizer.get_feature_names_out())
  tfidf_df.index = texts_to_correlate

  # Compute the cosine similarity between movie genres
  cosine_similarity_array = cosine_similarity(tfidf_df)

  # Get the indices of the similar movies based on cosine similarity
  cosine_similarity_df = pd.DataFrame(cosine_similarity_array, index=tfidf_df.index, columns=tfidf_df.index)
  student_reasoning_similarity_series = cosine_similarity_df[student_reasoning][1:]

  if (DEBUG_LEVEL > 0):
    print("student_reasoning_similarity_series:  %s" % str(student_reasoning_similarity_series))
    print("student_reasoning_similarity_series.max().size:  %s" % str(student_reasoning_similarity_series.max().size))
    print("student_reasoning_similarity_series.max().shape:  %s" % str(student_reasoning_similarity_series.max().shape))

  student_reasoning_similarity_percentage = student_reasoning_similarity_series.max()
  if (student_reasoning_similarity_series.max().size > 1):
    if (DEBUG_LEVEL > 0):
      print("student_reasoning_similarity_series.max().iloc[0]:  %s" % str(student_reasoning_similarity_series.max().iloc[0]))
    student_reasoning_similarity_percentage = student_reasoning_similarity_series.max().iloc[0]

  return student_reasoning_similarity_percentage


def pick_best_chatgpt_feedback(problem_df, student_reasoning, chatgpt_feedback_list, n):
  # This function uses TF-IDF and cosine similarity to find similarities
  # between student reasoning + teacher notes, against a list of chatgpt feedback n
  # items long.

  texts_to_correlate = [ student_reasoning ] 

  if ('related_properties' in problem_df.columns):
    related_properties_df = pd.DataFrame.from_dict(problem_df['related_properties']).dropna()
    if (len(related_properties_df) > 0):
      for index in range(1, len(related_properties_df)):
        property_text = "; %s:  %s" %  (related_properties_df['related_properties'][index]['property'], related_properties_df['related_properties'][index]['usage'])
        texts_to_correlate[0] += property_text

  if ('related_problems' in problem_df.columns):
    related_problems_df = pd.DataFrame.from_dict(problem_df['related_problems']).dropna()
    if (len(related_problems_df) > 0):
      for index in range(0, len(related_problems_df), 1):
        property_text = "; %s = %s" %  (related_problems_df['related_problems'][index]['problem'], related_problems_df['related_problems'][index]['answer'])
        texts_to_correlate[0] += property_text

  texts_to_correlate.extend(chatgpt_feedback_list)

  if (DEBUG_LEVEL > 0):
    print("Texts to correlate:  %s" % texts_to_correlate)

  vectorizer = TfidfVectorizer(min_df=2, max_df=0.7)
  vectorizer = TfidfVectorizer()
  vectorized_data = vectorizer.fit_transform(texts_to_correlate)

  tfidf_df = pd.DataFrame(vectorized_data.toarray(),
                          columns=vectorizer.get_feature_names_out())
  tfidf_df.index = texts_to_correlate

  # Compute the cosine similarity between movie genres
  cosine_similarity_array = cosine_similarity(tfidf_df)

  # Get the indices of the similar movies based on cosine similarity
  cosine_similarity_df = pd.DataFrame(cosine_similarity_array, index=tfidf_df.index, columns=tfidf_df.index)
  reasoning_similarity_series = cosine_similarity_df.iloc[0, 1:]
      
  if (DEBUG_LEVEL > 0):
    print("reasoning_similarity_series:  %s" % reasoning_similarity_series)
    print("reasoning_similarity_series.max():  %s" % reasoning_similarity_series.max())
    print("reasoning_similarity_series.idxmax():  %s" % reasoning_similarity_series.idxmax())

  return reasoning_similarity_series.idxmax()


In [245]:
import io
import json

previous_student_reasoning = []

if (DEBUG_LEVEL > 0):
  print(math_problems_df)

num_candidate_answers = math_problems_df.num_candidate_answers
retry_max_times = math_problems_df.retry_max_times
math_problems = math_problems_df.problems[0]

for math_problem in math_problems:
  # ---------------------------------------------------------------------------
  # The following fulfills bullet point 3(a) in the block comment above.
  #problem_df = pd.DataFrame.from_dict(math_problem)
  problem_df = pd.DataFrame({key: pd.Series(value) for key, value in math_problem.items()})
  student_answer = input("What is %s ?" % problem_df['problem'][0])
  # print("Problem:  %s, Answer:  %s, Student Answer:  %s" % (problem_df['problem'][0], problem_df['answer'][0], student_answer))

  # ---------------------------------------------------------------------------
  # The following fulfills bullet point 3(c) in the block comment above.
  student_reasoning = input("And what is your reasoning for your answer?")

  answer_correct = False

  if (problem_df['answer'][0] == student_answer):
    answer_correct = True

  if (not answer_correct):
    # ---------------------------------------------------------------------------
    # The following fulfills bullet point 3(b) in the block comment above.
    verification_query = "A student answered that %s = %s.  Verify the answer is incorrect, yes or no." % (problem_df['problem'][0], student_answer)
    # print(verification_query)
    if (CONNECT_TO_CHATGPT_API):  
      response = client.responses.create(
        model="gpt-5.5",
        instructions="You are a math teacher assistant.",
        input=verification_query
      )

      if (response.output_text[:3].lower() != "yes"):
        answer_correct = True

  if (not answer_correct):
    print(chatgpt_api_supplies_leading_questions(problem_df, student_answer, student_reasoning, num_candidate_answers[0]))
  else:
    print("Your answer was correct!")

  if (similarity_between_student_reasoning_and_properties(problem_df, student_reasoning, previous_student_answers) >= REASONING_THRESHOLD):
    print("Importantly, your reasoning was along the lines of what the teacher was looking for.")

  if (DEBUG_LEVEL > 0):
    print("previous_student_reasoning:  %s" % previous_student_reasoning)

  previous_student_reasoning.append(student_reasoning)


What is 4 * 25 ? 100
And what is your reasoning for your answer? 4 quarters are in a dollar


Your answer was correct!
Importantly, your reasoning was along the lines of what the teacher was looking for.


What is 4 * 250 ? 1000
And what is your reasoning for your answer? 4 * 250 = 4 * (25 * 10) = (4 * 25) * 10 = 100 * 10 = 1000


Your answer was correct!


What is 1000 / 4 ? 250
And what is your reasoning for your answer? 4 * 250 = 1000


Your answer was correct!
Importantly, your reasoning was along the lines of what the teacher was looking for.


What is 996 / 4 ? 6
And what is your reasoning for your answer? 9+9+6 = 24 and 24 / 4 = 6


If (24 div 4 = 6), what number are we dividing there: 24 or 996? Are those the same amount?


In [ ]:
# 4.  Say "thank you for the conversation" to the student and assure them that good elements of the math conversation
#     have been recorded (logged) and will be passed to a human teacher for review.  Allow the student to sign off the session.
#
print("Thank you for having this math conversation!  Your responses will be reviewed by human teachers to help you to improve.")


In [ ]:
# 5.  Disconnect from the ChatGPT API.
if (CONNECT_TO_CHATGPT_API):
  client.close()